# 🔬 Interactive Pandas Basics: OpenAQ India Air Quality EDA

Welcome to your first hands-on exploratory data analysis (EDA) notebook. Today we are exploring real air quality telemetry from **Indian CPCB monitoring stations** downloaded via the **OpenAQ v3 API**.

### 🎯 Learning Goals:
1. **Load and Inspect** a real-world ~105k row dataset.
2. **Subsetting & Querying** using clean Boolean indexing.
3. **Understanding Immutability** in Pandas (mastering `.copy()`).
4. **Finding & Cleaning Dirty Data** (handling nulls, zeroes, and negative readings).
5. **Split-Apply-Combine** via GroupBy to rank air pollution by station.

## 🛠️ Setup & Data Loading

In [2]:
import numpy as np
import pandas as pd

# Set display options to see more columns/rows cleanly
pd.set_option('display.max_columns', 15)
pd.set_option('display.width', 1000)

Let's load the raw CSV from `data/raw/india_aq_raw.csv`. 

> **Note:** In Jupyter, paths are relative to the notebook's directory. So we use `../data/raw/india_aq_raw.csv` to step up one directory first.

In [3]:
df = pd.read_csv("../data/raw/india_aq_raw.csv")
print(f"DataFrame loaded. Shape: {df.shape} (Rows, Columns)")

DataFrame loaded. Shape: (105265, 8) (Rows, Columns)


---
## 🔍 Step 1: Initial Data Inspection

Before diving into statistics, let's understand the schema. 
- `.dtypes` shows the datatype of each column.
- `.info()` shows count of non-null values and memory footprint.
- `.head()` and `.tail()` let us inspect the first/last records.

In [4]:
# 1. Display first 5 rows
df.head()

,location_id,location,sensor_id,parameter,units,value,datetime,datetime_local
0,17,R K Puram Delhi,12234783,no,ppb,93.2,2025-02-18T20:00:00Z,2025-02-19T01:30:00+05:30
1,17,R K Puram Delhi,12234783,no,ppb,90.4,2025-02-18T20:15:00Z,2025-02-19T01:45:00+05:30
2,17,R K Puram Delhi,12234783,no,ppb,84.7,2025-02-18T21:00:00Z,2025-02-19T02:30:00+05:30
3,17,R K Puram Delhi,12234783,no,ppb,80.8,2025-02-18T21:15:00Z,2025-02-19T02:45:00+05:30
4,17,R K Puram Delhi,12234783,no,ppb,81.6,2025-02-18T21:30:00Z,2025-02-19T03:00:00+05:30


In [ ]:
# 2. View column types & structural summary
df.info()

In [ ]:
# 3. Run basic statistical summaries
df.describe()

### 💡 Observations from `df.describe()`:
- Look at the `value` column (the pollutant concentration).
- Is the minimum value negative? (If so, why would a sensor record negative pollution? That is **dirty data**!).
- What is the 75th percentile vs. the maximum? A huge gap indicates potential **outliers**.

---
## 🎯 Step 2: Boolean Indexing & Subsetting

Boolean indexing is the core mechanism for filtering rows in Pandas. We create a Boolean mask (a Series of `True`/`False` values) and pass it to the DataFrame to subset it.

In [ ]:
# 1. Let's see what unique pollutants we have in the dataset
df["parameter"].unique()

In [ ]:
# 2. Let's filter for PM2.5 readings only
pm25_mask = df["parameter"] == "pm25"
df_pm25 = df[pm25_mask]
print(f"Found {len(df_pm25)} PM2.5 readings.")
df_pm25.head()

### ⚠️ Immutability Alert: The `SettingWithCopyWarning`

If you try to modify a sliced DataFrame directly (like adding a column to `df_pm25`), Pandas might yell at you with a `SettingWithCopyWarning` because it doesn't know if `df_pm25` is a view of the original `df` or a copy.

**The Rule:** Always use `.copy()` when slicing a subset of data that you intend to modify later!

In [ ]:
# Safe practice: Create an explicit copy
delhi_df = df[df["location"].str.contains("Delhi")].copy()
delhi_df["is_delhi"] = True
delhi_df.head()

---
## 🧹 Step 3: Cleaning Dirty Data

Real-world data is messy. Let's find missing values, zeros, and negative sensor telemetry.

In [ ]:
# 1. Count missing values in each column
df.isna().sum()

In [ ]:
# 2. Find invalid negative concentrations
negatives = df[df["value"] < 0]
print(f"Number of negative values: {len(negatives)}")
if len(negatives) > 0:
    print("Preview of negative readings:")
    print(negatives[["location", "parameter", "value", "units"]].head())

In [ ]:
# 3. Let's clean the dataset by removing negative readings and keeping only values >= 0
df_clean = df[df["value"] >= 0].copy()
print(f"Cleaned dataset size: {df_clean.shape[0]} rows (removed {len(negatives)} rows)")

---
## 📊 Step 4: Split-Apply-Combine (GroupBy)

Now let's compute real insights! What are the average PM2.5 concentrations at each station? 
We use `.groupby()` to split by station, grab the `value` column, and run `.mean()`.

In [ ]:
# Group PM2.5 values by location and sort descending
pm25_by_station = df_clean[df_clean["parameter"] == "pm25"].groupby("location")["value"].mean()
pm25_by_station.sort_values(ascending=False)

### 🚀 Multi-Aggregation Grouping
We can use `.agg()` to calculate multiple statistics at once (count of readings, mean, min, max, and std).

In [ ]:
station_stats = df_clean[df_clean["parameter"] == "pm25"].groupby("location")["value"].agg(
    ["count", "mean", "min", "max", "std"]
)
station_stats.sort_values(by="mean", ascending=False)

---
## ⏰ Step 5: Introduction to Datetime Parsing

Currently, our `datetime` column is represented as text strings (type `object`). For time-series analysis, we must parse them into real Pandas `datetime64` types.

In [ ]:
# Convert datetime column to datetime objects
df_clean["datetime"] = pd.to_datetime(df_clean["datetime"])
print(f"Datetime column parsed! New type: {df_clean['datetime'].dtype}")

In [ ]:
# Extract temporal features
df_clean["hour"] = df_clean["datetime"].dt.hour
df_clean["day_name"] = df_clean["datetime"].dt.day_name()

# Look at the raw records with new temporal columns
df_clean[["location", "parameter", "value", "hour", "day_name"]].head()

---
## 🏆 Challenge Exercises for You:

1. **NO2 Pollution:** Find which monitoring station has the single highest **NO2** reading.
2. **Unhealthy Days:** How many readings in this dataset have a PM2.5 level greater than **150 µg/m³** (considered hazardous)?
3. **Telemetry Gaps:** Which monitoring station has the lowest number of total readings (least active)?
4. **Hourly Cycles:** Group by `hour` and find which hour of the day has the highest average PM2.5 concentration across all stations.